# Person 2 — Class SLM RAG experiment (Colab CPU)
Choose a **CPU** runtime. This is a tiny class-trained character model, not a Hugging Face model. Upload the code bundle and then the exact private `selected_baseline.pt` in the separate checkpoint cell. No Ollama server, Groq key, training or weights download is required.

The class notebook's TF-IDF (English stop words; unigrams/bigrams) and cosine retrieval are retained. The generator is replaced with Person 1's original SLM. `min_score=0` keeps exactly K documents; rank order is unchanged. Prompts use document text and question separated by newlines because citation brackets and uppercase labels are outside the model vocabulary. See `a7/person2/class_rag_provenance.json` and `a7/slm/class_rag_source.ipynb`.

Five fictional household facts absent from the training corpus and five fixed questions are shared with Persons 3/4. We test baseline plus K=1,2,3,5: **5 baseline + 20 RAG continuations**. Do not change architecture, embeddings, vocabulary or the original checkpoint to improve results.

In [1]:
from google.colab import files
uploaded = files.upload()  # Select hestia-a7-person2-colab.zip only.
import zipfile
from pathlib import Path
import io
assert len(uploaded) == 1, "Select only the Colab bundle ZIP."
bundle_name, bundle_data = next(iter(uploaded.items()))
assert bundle_name.lower().endswith(".zip"), "Upload the Colab bundle ZIP."
with zipfile.ZipFile(io.BytesIO(bundle_data)) as archive:
    archive.extractall("/content")
%cd /content/hestia-a7-person2

Saving hestia-a7-person2-colab.zip to hestia-a7-person2-colab.zip
/content/hestia-a7-person2


In [2]:
%pip install -r a7/requirements-person2-rag.txt

In [3]:
from google.colab import files
uploaded_checkpoint = files.upload()  # Select only selected_baseline.pt.
assert "selected_baseline.pt" in uploaded_checkpoint
checkpoint = Path("a7/slm/checkpoints/selected_baseline.pt")
checkpoint.parent.mkdir(parents=True, exist_ok=True)
checkpoint.write_bytes(uploaded_checkpoint["selected_baseline.pt"])
import json
from a7.slm.rag_experiment import verify_checkpoint
manifest = json.loads(Path("a7/slm/results/selected_checkpoint.json").read_text())
verify_checkpoint(checkpoint, manifest)
print("Exact Person 1 checkpoint file verified:", manifest["checkpoint_sha256"])

Saving selected_baseline.pt to selected_baseline.pt
Exact Person 1 checkpoint file verified: ea5b43d934bed646d02183c50e371f89df57801cc1bc7ef45027ccb2d8de3f36


In [4]:
from a7.slm.rag_experiment import run_experiments
result_dir = Path("a7/slm/results/person2_rag_colab")
# Refuses to overwrite existing evidence. For another run choose a new directory.
result = run_experiments(checkpoint, result_dir)
print("Weights unchanged:", result["weights_unchanged"])
print("Environment:", result["environment"])

Q1 baseline 0.202s
Q1 top-1 0.074s
Q1 top-2 0.070s
Q1 top-3 0.070s
Q1 top-5 0.069s
Q2 baseline 0.101s
Q2 top-1 0.068s
Q2 top-2 0.076s
Q2 top-3 0.070s
Q2 top-5 0.083s
Q3 baseline 0.183s
Q3 top-1 0.594s
Q3 top-2 0.214s
Q3 top-3 0.066s
Q3 top-5 0.067s
Q4 baseline 0.073s
Q4 top-1 0.070s
Q4 top-2 0.070s
Q4 top-3 0.074s
Q4 top-5 0.072s
Q5 baseline 0.068s
Q5 top-1 0.069s
Q5 top-2 0.088s
Q5 top-3 0.068s
Q5 top-5 0.067s
Weights unchanged: True
Environment: {'python': '3.13.16', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.39', 'packages': {'torch': '2.11.0+cpu', 'numpy': '2.1.3', 'scikit-learn': '1.6.1', 'pandas': '2.2.3'}}


In [5]:
import pandas as pd
baseline = pd.read_csv(result_dir / "person2_baseline.csv", keep_default_na=False)
rag = pd.read_csv(result_dir / "person2_rag.csv", keep_default_na=False)
assert len(baseline) == 5 and len(rag) == 20
assert result["original_state_sha256"] == result["final_state_sha256"] == manifest["state_sha256"]
pd.set_option("display.max_colwidth", None)
display(baseline[["question_id", "question", "response"]])
display(rag[["question_id", "top_k", "retrieved_document_ids", "visible_context", "expected_facts_visible", "response"]])

,question_id,question,response
0,Q1,when is laundry day.,the predicts probabmenters are use models and un learning res artifining paplications.\nstudents crea
1,Q2,when is trash day.,the predicts probabilition paransmelld django application artificial intelligencers usefuse models l
2,Q3,where are guest towels.,artificial inngual networks are pytok\nstentraining hang helps artificial intelligencers artuage mode
3,Q4,where are spare bulbs.,attention rkeach.\ntrach.\ntraning he students useful softworks learne predicts artuage models can re
4,Q5,where is dish soap.,students expe models chartificial intelligencers con prevers artificial intelligencers.\nattion usegr


,question_id,top_k,retrieved_document_ids,visible_context,expected_facts_visible,response
0,Q1,1,"[""D1""]",is monday.\nwhen is laundry day.\n,"{""monday"": true}",smalll learning res models preveows.\ningrametion learaning helps useful models models can learnung o
1,Q1,2,"[""D1"", ""D2""]",is friday.\nwhen is laundry day.\n,"{""monday"": false}",smalll learning res models preveows.\ningrametion learaning helps useful models models can learnung o
2,Q1,3,"[""D1"", ""D2"", ""D5""]",the sink.\nwhen is laundry day.\n,"{""monday"": false}",smalll learning res models preveows.\ningrametion learaning helps useful models models can learnung o
3,Q1,5,"[""D1"", ""D2"", ""D5"", ""D3"", ""D4""]",the shed.\nwhen is laundry day.\n,"{""monday"": false}",smalll learning res models preveows.\ningrametion learaning helps useful models models can learnung o
4,Q2,1,"[""D2""]",y is friday.\nwhen is trash day.\n,"{""friday"": true}",the predicts probabilition paransmelld django application artificial intelligencers usefuse models l
5,Q2,2,"[""D2"", ""D1""]",y is monday.\nwhen is trash day.\n,"{""friday"": false}",the predicts probabilition paransmelld django application artificial intelligencers usefuse models l
6,Q2,3,"[""D2"", ""D1"", ""D5""]",er the sink.\nwhen is trash day.\n,"{""friday"": false}",the predicts probabilition paransmelld django application artificial intelligencers usefuse models l
7,Q2,5,"[""D2"", ""D1"", ""D5"", ""D3"", ""D4""]",in the shed.\nwhen is trash day.\n,"{""friday"": false}",the predicts probabilition paransmelld django application artificial intelligencers usefuse models l
8,Q3,1,"[""D3""]",e hall.\nwhere are guest towels.\n,"{""in the hall"": false}",artition use preversitity lon learnedicts artual ingaal networks learnedicts artificial inngerameter
9,Q3,2,"[""D3"", ""D5""]",e sink.\nwhere are guest towels.\n,"{""in the hall"": false}",artition use preversitity lon learnedicts artual ingaal networks learnedicts artificial inngerameter


## Manual response review (required before final submission)
Read the **continuation only**, not facts copied into the prompt. For every baseline and RAG response, score overall `response_quality`, `factual_accuracy` and `relevance` from 1–10; mark `unsupported_information` true/false and explain your decision. Unsupported/gibberish output can legitimately score poorly. Do not invent successful answers.

Edit each tuple below as `(quality, accuracy, relevance, unsupported, notes)`. All values start unset. The next cell deliberately refuses to export a falsely completed review. The original raw evidence is preserved separately.

In [6]:
reviews = {(q["question_id"], k): (None, None, None, None, "")
           for q in result["questions"] for k in (0, 1, 2, 3, 5)}
# Replace each value with your actual assessment, e.g. reviews[("Q1", 0)] = (...)

In [9]:
# AI-assisted review of the supplied 25 raw continuations; inspect before submitting.
# Run this AFTER the reviews-initialization cell, BEFORE the review-export cell.
reviews = {
    ('Q1', 0): (1, 1, 1, True, 'Unrelated fragments about learning and models; no laundry-day answer. Expected fact: monday. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q1', 1): (1, 1, 1, True, 'Unrelated fragments about learning and models; no laundry-day answer. Expected fact: monday. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q1', 2): (1, 1, 1, True, 'Unrelated fragments about learning and models; no laundry-day answer. Expected fact: monday. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q1', 3): (1, 1, 1, True, 'Unrelated fragments about learning and models; no laundry-day answer. Expected fact: monday. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q1', 5): (1, 1, 1, True, 'Unrelated fragments about learning and models; no laundry-day answer. Expected fact: monday. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q2', 0): (1, 1, 1, True, 'Unrelated fragments about probability, Django and models; no trash-day answer. Expected fact: friday. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q2', 1): (1, 1, 1, True, 'Unrelated fragments about probability, Django and models; no trash-day answer. Expected fact: friday. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q2', 2): (1, 1, 1, True, 'Unrelated fragments about probability, Django and models; no trash-day answer. Expected fact: friday. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q2', 3): (1, 1, 1, True, 'Unrelated fragments about probability, Django and models; no trash-day answer. Expected fact: friday. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q2', 5): (1, 1, 1, True, 'Unrelated fragments about probability, Django and models; no trash-day answer. Expected fact: friday. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q3', 0): (1, 1, 1, True, 'Unrelated fragments about neural networks and training; no towel location. Expected fact: in the hall. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q3', 1): (1, 1, 1, True, 'Unrelated fragments about neural networks and training; no towel location. Expected fact: in the hall. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q3', 2): (1, 1, 1, True, 'Unrelated fragments about neural networks and training; no towel location. Expected fact: in the hall. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q3', 3): (1, 1, 1, True, 'Unrelated fragments about neural networks and training; no towel location. Expected fact: in the hall. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q3', 5): (1, 1, 1, True, 'Unrelated fragments about neural networks and training; no towel location. Expected fact: in the hall. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q4', 0): (1, 1, 1, True, 'Unrelated fragments about attention, students and models; no bulb location. Expected fact: in the shed. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q4', 1): (1, 1, 1, True, 'Unrelated fragments about attention, students and models; no bulb location. Expected fact: in the shed. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q4', 2): (1, 1, 1, True, 'Unrelated fragments about attention, students and models; no bulb location. Expected fact: in the shed. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q4', 3): (1, 1, 1, True, 'Unrelated fragments about attention, students and models; no bulb location. Expected fact: in the shed. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q4', 5): (1, 1, 1, True, 'Unrelated fragments about attention, students and models; no bulb location. Expected fact: in the shed. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q5', 0): (1, 1, 1, True, 'Unrelated fragments about students, software or learning; no soap location. Expected fact: under the sink. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q5', 1): (1, 1, 1, True, 'Unrelated fragments about students, software or learning; no soap location. Expected fact: under the sink. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q5', 2): (1, 1, 1, True, 'Unrelated fragments about students, software or learning; no soap location. Expected fact: under the sink. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q5', 3): (1, 1, 1, True, 'Unrelated fragments about students, software or learning; no soap location. Expected fact: under the sink. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
    ('Q5', 5): (1, 1, 1, True, 'Unrelated fragments about students, software or learning; no soap location. Expected fact: under the sink. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.'),
}
assert len(reviews) == 25


In [10]:
import copy
reviewed = copy.deepcopy(result)
for q in reviewed["questions"]:
    for k, response in [(0, q["baseline"])] + [(r["top_k"], r) for r in q["rag"]]:
        quality, accuracy, relevance, unsupported, notes = reviews[(q["question_id"], k)]
        assert type(quality) is int and 1 <= quality <= 10, "Finish every overall quality score."
        assert type(accuracy) is int and 1 <= accuracy <= 10, "Finish every accuracy score."
        assert type(relevance) is int and 1 <= relevance <= 10, "Finish every relevance score."
        assert type(unsupported) is bool and notes.strip(), "Complete flags and evidence-based notes."
        response.update(response_quality=quality, factual_accuracy=accuracy, relevance=relevance,
                        unsupported_information=unsupported, review_notes=notes)
reviewed["review_complete"] = True
(result_dir / "person2_rag_reviewed.json").write_text(json.dumps(reviewed, indent=2))
print("Reviewed Person 2 portion saved; Person 3 fine-tuned responses must be merged later by Person 4.")

Reviewed Person 2 portion saved; Person 3 fine-tuned responses must be merged later by Person 4.


## Required analysis — answer each in one or two sentences after the run
1. Did providing retrieved information change the SLM’s responses?
2. Did RAG improve factual accuracy?
3. Did retrieving more documents always improve the responses?
4. How did the SLM’s context-length limitation affect the results?
5. Why might RAG behave differently with a small character-level model compared with a larger pretrained language model?

Support these answers with the actual response table and reviewed scores. Separate retrieval success from generation success. Inspect `visible_context` and `expected_facts_visible`: the original model crops to the final **32 characters**, so retrieved facts outside that window are unavailable. Identical visible contexts may generate identical answers at the fixed seed. Failed grounding is a valid finding; do not present it as successful RAG.

Share the KB/questions, prompt packing, CPU threads=2, seed=42, temperature=.7 and 100 **character** tokens with Person 3 before fine-tuning. Use identical Colab package versions and verify matching pre-training baseline outputs. The same tensor hash alone does not establish cross-version generation equivalence. The Person 2 export does not fabricate Person 3's fine-tuned results.

This notebook is setup until executed, reviewed and these answers are completed.

1. **Did providing retrieved information change the SLM's responses?** Yes: Top-1 changes the continuation for Q1, Q3, Q4 and Q5, while Q2 remains identical to baseline. These changes do not produce correct household answers.
2. **Did RAG improve factual accuracy?** No: all five baseline answers and all twenty RAG answers fail to state the expected fact. The reviewed factual-accuracy score remains 1/10 in every condition.
3. **Did retrieving more documents always improve the responses?** No: Top-1 and Top-2 produce identical continuations for all five questions; Q5 changes at Top-3, and Top-5 matches Top-3 for every question. No K setting gives a correct answer, so the experiment establishes no quality advantage from a larger K.
4. **How did the SLM's context-length limitation affect the results?** The last-32-character crop removes large portions of the retrieved text; for example Q1 Top-1 retains “is monday.” but Top-2 retains “is friday.” from a distractor. The full expected facts for the towel, bulb and soap locations are already cut off even at Top-1, limiting what the model can use.
5. **Why might RAG behave differently with a small character-level model compared with a larger pretrained language model?** This class SLM learned short character patterns from a small generic corpus and is not an instruction-trained question-answering model, so supplying facts does not ensure it can use them. A larger pretrained instruction model would generally have more learned language skills and a longer token context, but that comparison was not measured here.

In [12]:
import shutil
archive_path = shutil.make_archive("/content/person2_rag_evidence", "zip", root_dir=result_dir)
files.download(archive_path)
# Also File → Download → Download .ipynb to retain outputs in the executed notebook.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Saved evidence tables

These tables render the five knowledge documents and the 25 **saved, reviewed** model continuations. K=0 is the baseline with no retrieved documents. Scores are the disclosed AI-assisted assessments above; inspect the raw responses and notes before submitting. No model is run by these display cells.

In [13]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

documents = json.loads(Path("a7/slm/knowledge_base/documents.json").read_text())
assert len(documents) == 5 and len({doc["id"] for doc in documents}) == 5
display(pd.DataFrame(documents)[["id", "title", "text"]])

id,title,text
D1,laundry guide,laundry day is monday.
D2,trash guide,trash day is friday.
D3,guest towels,guest towels are in the hall.
D4,spare bulbs,spare bulbs are in the shed.
D5,dish soap,dish soap is under the sink.


In [14]:
reviewed_saved = json.loads((result_dir / "person2_rag_reviewed.json").read_text())
assert reviewed_saved["review_complete"] is True
by_id = {doc["id"]: doc for doc in documents}
comparison_rows = []
for item in reviewed_saved["questions"]:
    for k, answer in [(0, item["baseline"])] + [(row["top_k"], row) for row in item["rag"]]:
        ids = answer.get("retrieved_document_ids", [])
        comparison_rows.append({
            "question": item["question"],
            "K": k,
            "retrieved documents": "; ".join(f"{doc_id}: {by_id[doc_id]['text']}" for doc_id in ids) or "None (baseline)",
            "response": answer["response"],
            "reviewed quality (1–10)": answer["response_quality"],
        })
comparison = pd.DataFrame(comparison_rows)
assert len(comparison) == 25
assert set(comparison["K"]) == {0, 1, 2, 3, 5}
pd.set_option("display.max_rows", 30)
pd.set_option("display.max_colwidth", None)
display(comparison)

question,K,retrieved documents,response,reviewed quality (1–10)
when is laundry day.,0,None (baseline),the predicts probabmenters are use models and un learning res artifining paplications.\nstudents crea,1
when is laundry day.,1,D1: laundry day is monday.,smalll learning res models preveows.\ningrametion learaning helps useful models models can learnung o,1
when is laundry day.,2,D1: laundry day is monday.; D2: trash day is friday.,smalll learning res models preveows.\ningrametion learaning helps useful models models can learnung o,1
when is laundry day.,3,D1: laundry day is monday.; D2: trash day is friday.; D5: dish soap is under the sink.,smalll learning res models preveows.\ningrametion learaning helps useful models models can learnung o,1
when is laundry day.,5,D1: laundry day is monday.; D2: trash day is friday.; D5: dish soap is under the sink.; D3: guest towels are in the hall.; D4: spare bulbs are in the shed.,smalll learning res models preveows.\ningrametion learaning helps useful models models can learnung o,1
when is trash day.,0,None (baseline),the predicts probabilition paransmelld django application artificial intelligencers usefuse models l,1
when is trash day.,1,D2: trash day is friday.,the predicts probabilition paransmelld django application artificial intelligencers usefuse models l,1
when is trash day.,2,D2: trash day is friday.; D1: laundry day is monday.,the predicts probabilition paransmelld django application artificial intelligencers usefuse models l,1
when is trash day.,3,D2: trash day is friday.; D1: laundry day is monday.; D5: dish soap is under the sink.,the predicts probabilition paransmelld django application artificial intelligencers usefuse models l,1
when is trash day.,5,D2: trash day is friday.; D1: laundry day is monday.; D5: dish soap is under the sink.; D3: guest towels are in the hall.; D4: spare bulbs are in the shed.,the predicts probabilition paransmelld django application artificial intelligencers usefuse models l,1
